# Lab 07: HOG + SVM vs GA-HOG + SVM (BloodMNIST)
Cells ko upar se neeche ek ek karke run karo (Runtime → Run all bhi chalega).

## Cell 1: Install

In [ ]:
!pip install -q medmnist scikit-image tabulate

## Cell 2: Imports aur settings

In [ ]:
import json, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from joblib import Parallel, delayed
from skimage.feature import hog
from skimage.transform import resize
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             balanced_accuracy_score, cohen_kappa_score,
                             matthews_corrcoef, confusion_matrix)

# ---------------- SETTINGS (yahan se badlo) ----------------
SMOKE      = False          # True = chhota synthetic test (2-3 min), False = asli run
DATASET    = "bloodmnist"   # ya "pneumoniamnist" (tez, binary)
SIZE       = 128            # 128 (lab requirement) ya 64 (tez)
GENS       = 15             # GA generations
GA_TRAIN_N = 2000           # GA fitness ke liye training subset (speed)
SEED       = 42
OUT        = "results_lab07"
# -----------------------------------------------------------
os.makedirs(OUT, exist_ok=True)
np.random.seed(SEED)

## Cell 3: Dataset load (train / val / test split)

In [ ]:
def load_medmnist(name, size):
    import medmnist
    from medmnist import INFO
    cls = getattr(medmnist, INFO[name]["python_class"])
    sets = {}
    for split in ["train", "val", "test"]:
        try:
            ds = cls(split=split, download=True, size=size)
        except TypeError:
            ds = cls(split=split, download=True)
        sets[split] = (ds.imgs, ds.labels.ravel())
    return sets

def make_synthetic(n_per_class=60, size=64, n_classes=3):
    rng = np.random.RandomState(SEED)
    def gen(n):
        X, y = [], []
        for c in range(n_classes):
            for _ in range(n):
                xx, yy = np.meshgrid(np.arange(size), np.arange(size))
                ang = c * np.pi / n_classes
                img = np.sin((xx * np.cos(ang) + yy * np.sin(ang)) / 3.0)
                img = (img - img.min()) / (img.max() - img.min()) + rng.rand(size, size) * 0.8
                X.append((img / img.max() * 255).astype(np.uint8)); y.append(c)
        return np.array(X), np.array(y)
    return {"train": gen(n_per_class), "val": gen(n_per_class // 2), "test": gen(n_per_class // 2)}

if SMOKE:
    data, RES, GENS, GA_N = make_synthetic(), 64, 4, 10**9
    CONFIGS = [dict(pop_size=6, p_mut=0.005), dict(pop_size=8, p_mut=0.02)]
else:
    data, RES, GA_N = load_medmnist(DATASET, SIZE), SIZE, GA_TRAIN_N
    CONFIGS = [dict(pop_size=20, p_mut=0.001), dict(pop_size=30, p_mut=0.005)]

(Xtr_i, ytr), (Xva_i, yva), (Xte_i, yte) = data["train"], data["val"], data["test"]
print(f"train={len(ytr)}  val={len(yva)}  test={len(yte)}  classes={len(np.unique(ytr))}")

## Cell 4: HOG feature extraction (baseline aur GA dono ke liye same)

In [ ]:
def to_gray(img):
    if img.ndim == 3:
        img = img.mean(axis=2)
    return img.astype(np.float32) / 255.0

def hog_one(img, res):
    g = to_gray(img)
    if g.shape[0] != res:
        g = resize(g, (res, res), anti_aliasing=True)
    return hog(g, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2),
               block_norm="L2-Hys", feature_vector=True)

def extract(X, res):
    return np.array(Parallel(n_jobs=-1)(delayed(hog_one)(im, res) for im in X), dtype=np.float32)

t0 = time.time()
Xtr, Xva, Xte = extract(Xtr_i, RES), extract(Xva_i, RES), extract(Xte_i, RES)
t_hog = time.time() - t0
n_feat = Xtr.shape[1]
print(f"HOG features per image: {n_feat}   (time {t_hog:.1f}s)")

## Cell 5: Metrics function

In [ ]:
def full_metrics(y, p):
    cm = confusion_matrix(y, p)
    tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp; tn = cm.sum() - tp - fp - fn
    spec = np.mean(tn / np.maximum(tn + fp, 1))   # macro-averaged one-vs-rest specificity
    return {
        "Accuracy (%)": 100 * accuracy_score(y, p),
        "Precision (Macro, %)": 100 * precision_score(y, p, average="macro", zero_division=0),
        "Recall (Macro, %)": 100 * recall_score(y, p, average="macro", zero_division=0),
        "Macro-F1 (%)": 100 * f1_score(y, p, average="macro", zero_division=0),
        "Weighted-F1 (%)": 100 * f1_score(y, p, average="weighted", zero_division=0),
        "Balanced accuracy (%)": 100 * balanced_accuracy_score(y, p),
        "Specificity (Macro, %)": 100 * spec,
        "Cohen's kappa": cohen_kappa_score(y, p),
        "Matthews corr. coef.": matthews_corrcoef(y, p),
    }

GRID = [(C, "scale") for C in (1, 10, 100)]

def tune_svm(Xtr, ytr, Xva, yva):
    """C validation macro-F1 se tune hota hai."""
    best = (-1, None)
    for C, g in GRID:
        clf = SVC(kernel="rbf", C=C, gamma=g, random_state=SEED).fit(Xtr, ytr)
        f1 = f1_score(yva, clf.predict(Xva), average="macro", zero_division=0)
        if f1 > best[0]:
            best = (f1, (C, g))
    return best

## Cell 6: Baseline HOG + SVM

In [ ]:
t0 = time.time()
f1v, (C, g) = tune_svm(Xtr, ytr, Xva, yva)
base = SVC(kernel="rbf", C=C, gamma=g, random_state=SEED).fit(Xtr, ytr)
t_base_train = time.time() - t0

t0 = time.time(); p_base = base.predict(Xte); t_base_pred = time.time() - t0
m_base = full_metrics(yte, p_base)
print(f"Baseline: best C={C}, val macro-F1={f1v:.4f}, test macro-F1={m_base['Macro-F1 (%)']:.2f}")

## Cell 7: Genetic Algorithm class

In [ ]:
class GA:
    def __init__(self, Xtr, ytr, Xva, yva, pop_size, p_mut, p_cx=0.8, gens=15,
                 lam=0.05, init_p=0.3, tourn=3, elite=2, seed=SEED):
        self.Xtr, self.ytr, self.Xva, self.yva = Xtr, ytr, Xva, yva
        self.n = Xtr.shape[1]
        self.pop_size, self.p_mut, self.p_cx, self.gens = pop_size, p_mut, p_cx, gens
        self.lam, self.init_p, self.tourn, self.elite = lam, init_p, tourn, elite
        self.rng = np.random.RandomState(seed)

    def fitness(self, mask):
        k = mask.sum()
        if k < 10:
            return -1.0, 0.0
        clf = SVC(kernel="rbf", C=10, gamma="scale", random_state=SEED)
        clf.fit(self.Xtr[:, mask], self.ytr)
        f1 = f1_score(self.yva, clf.predict(self.Xva[:, mask]), average="macro", zero_division=0)
        return f1 - self.lam * (k / self.n), f1      # fitness = macro-F1 - penalty * feature ratio

    def evaluate(self, pop):
        res = Parallel(n_jobs=-1)(delayed(self.fitness)(m) for m in pop)
        return np.array([r[0] for r in res]), np.array([r[1] for r in res])

    def select(self, pop, fit):                       # tournament selection
        idx = self.rng.randint(0, len(pop), self.tourn)
        return pop[idx[np.argmax(fit[idx])]].copy()

    def crossover(self, a, b):                        # uniform crossover
        if self.rng.rand() < self.p_cx:
            m = self.rng.rand(self.n) < 0.5
            return np.where(m, a, b), np.where(m, b, a)
        return a.copy(), b.copy()

    def mutate(self, c):                              # bit-flip mutation
        flip = self.rng.rand(self.n) < self.p_mut
        c[flip] = ~c[flip]
        return c

    def run(self):
        pop = self.rng.rand(self.pop_size, self.n) < self.init_p     # population initialization
        hist = {"best": [], "mean": [], "best_f1": [], "best_nfeat": []}
        for g in range(self.gens):
            fit, f1s = self.evaluate(pop)
            b = np.argmax(fit)
            hist["best"].append(float(fit[b])); hist["mean"].append(float(fit.mean()))
            hist["best_f1"].append(float(f1s[b])); hist["best_nfeat"].append(int(pop[b].sum()))
            print(f"  gen {g+1:02d}/{self.gens}  best fit={fit[b]:.4f}  val F1={f1s[b]:.4f}  features={pop[b].sum()}")
            order = np.argsort(-fit)
            new = [pop[i].copy() for i in order[:self.elite]]        # elitism
            while len(new) < self.pop_size:
                c1, c2 = self.crossover(self.select(pop, fit), self.select(pop, fit))
                new += [self.mutate(c1), self.mutate(c2)]
            pop = np.array(new[:self.pop_size])
        fit, f1s = self.evaluate(pop)
        b = np.argmax(fit)
        return pop[b], float(fit[b]), hist

## Cell 8: GA chalao (2 settings) - ye sabse lamba cell hai

In [ ]:
rng = np.random.RandomState(SEED)
sub = rng.choice(len(ytr), min(GA_N, len(ytr)), replace=False)

ga_runs = []
for i, cfg in enumerate(CONFIGS):
    print(f"\nGA setting {i+1}: {cfg}")
    t0 = time.time()
    ga = GA(Xtr[sub], ytr[sub], Xva, yva, gens=GENS, **cfg)
    mask, best_fit, hist = ga.run()
    ga_runs.append(dict(cfg=cfg, mask=mask, fit=best_fit, hist=hist, time=time.time() - t0))

## Cell 9: Best GA setting chuno (validation se) aur final test evaluation

In [ ]:
best = max(ga_runs, key=lambda r: r["fit"])      # selection sirf validation fitness se
mask = best["mask"]

t0 = time.time()
f1v2, (C2, g2) = tune_svm(Xtr[:, mask], ytr, Xva[:, mask], yva)
ga_clf = SVC(kernel="rbf", C=C2, gamma=g2, random_state=SEED).fit(Xtr[:, mask], ytr)
t_ga_train = time.time() - t0

t0 = time.time(); p_ga = ga_clf.predict(Xte[:, mask]); t_ga_pred = time.time() - t0
m_ga = full_metrics(yte, p_ga)
print(f"Selected GA setting: {best['cfg']}   features {n_feat} -> {int(mask.sum())}")
print(f"GA-HOG test macro-F1 = {m_ga['Macro-F1 (%)']:.2f}   (baseline {m_base['Macro-F1 (%)']:.2f})")

## Cell 10: Tables (Table 1 to 4)

In [ ]:
t1 = pd.DataFrame([
    ["Medical dataset", DATASET if not SMOKE else "synthetic (smoke test)", "Same dataset"],
    ["Image resolution", f"{RES} x {RES}", f"{RES} x {RES}"],
    ["HOG orientations", 9, 9], ["Pixels per cell", "8 x 8", "8 x 8"],
    ["Cells per block", "2 x 2", "2 x 2"], ["Block normalization", "L2-Hys", "L2-Hys"],
    ["Feature selection", "None", "Genetic Algorithm"],
    ["Classifier / kernel", "SVM / RBF", "SVM / RBF"],
    ["Hyperparameter tuning", f"Validation set (C={C})", f"Validation set (C={C2})"],
    ["Data split", "Identical fixed split", "Identical fixed split"],
    ["Random seed", SEED, SEED],
    ["Number of features", n_feat, int(mask.sum())],
], columns=["Parameter", "Baseline: HOG + SVM", "Proposed: GA-HOG + SVM"])

t2 = pd.DataFrame([[k, round(m_base[k], 4), round(m_ga[k], 4), round(m_ga[k] - m_base[k], 4)] for k in m_base],
                  columns=["Metric", "HOG + SVM (Baseline)", "GA-HOG + SVM (Proposed)", "Improvement"])

t3 = pd.DataFrame([[r["cfg"]["pop_size"], r["cfg"]["p_mut"], GENS, round(r["fit"], 4),
                    int(r["mask"].sum()), round(r["time"], 1)] for r in ga_runs],
                  columns=["Population", "Mutation prob.", "Generations", "Best val fitness",
                           "Selected features", "GA time (s)"])

t4 = pd.DataFrame([
    ["HOG extraction (s)", round(t_hog, 1), round(t_hog, 1)],
    ["GA search (s)", 0, round(sum(r["time"] for r in ga_runs), 1)],
    ["SVM tuning + training (s)", round(t_base_train, 1), round(t_ga_train, 1)],
    ["Test prediction (s)", round(t_base_pred, 2), round(t_ga_pred, 2)],
    ["Number of features", n_feat, int(mask.sum())],
    ["Feature reduction (%)", 0, round(100 * (1 - mask.sum() / n_feat), 1)],
], columns=["Item", "Baseline", "GA-HOG"])

for name, t in [("Table 1. Configuration", t1), ("Table 2. Test performance", t2),
                ("Table 3. GA settings (validation)", t3), ("Table 4. Computational cost", t4)]:
    print("\n" + name)
    display(t)
    t.to_csv(f"{OUT}/{name.split('.')[0].lower().replace(' ', '')}.csv", index=False)

## Cell 11: Plots

In [ ]:
# Plot 1: GA fitness across generations
plt.figure(figsize=(7, 4.5))
for r in ga_runs:
    lab = f"pop={r['cfg']['pop_size']}, mut={r['cfg']['p_mut']}"
    plt.plot(range(1, GENS + 1), r["hist"]["best"], marker="o", label=f"best ({lab})")
    plt.plot(range(1, GENS + 1), r["hist"]["mean"], ls="--", alpha=.6, label=f"mean ({lab})")
plt.xlabel("Generation"); plt.ylabel("Fitness (val macro-F1 - penalty)")
plt.title("GA fitness across generations"); plt.legend(fontsize=7); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{OUT}/ga_fitness.png", dpi=200); plt.show()

# Plot 2: number of features before / after
plt.figure(figsize=(4.5, 4))
b = plt.bar(["HOG\n(baseline)", "GA-HOG"], [n_feat, int(mask.sum())], color=["#888", "#2a7"])
plt.bar_label(b); plt.ylabel("Number of features"); plt.title("Feature count")
plt.tight_layout(); plt.savefig(f"{OUT}/feature_count.png", dpi=200); plt.show()

# Plot 3: performance comparison
labels = ["Accuracy (%)", "Macro-F1 (%)", "Balanced accuracy (%)"]
x = np.arange(len(labels)); w = .35
plt.figure(figsize=(6, 4))
plt.bar(x - w/2, [m_base[l] for l in labels], w, label="HOG + SVM")
plt.bar(x + w/2, [m_ga[l] for l in labels], w, label="GA-HOG + SVM")
plt.xticks(x, labels, fontsize=8); plt.legend(); plt.ylabel("%"); plt.title("Test performance")
plt.tight_layout(); plt.savefig(f"{OUT}/performance_comparison.png", dpi=200); plt.show()

## Cell 12: Results zip download karo (report ke liye)

In [ ]:
import shutil
json.dump({"baseline": m_base, "ga_hog": m_ga, "n_features": [n_feat, int(mask.sum())],
           "selected_ga": best["cfg"]}, open(f"{OUT}/results.json", "w"), indent=2)
np.save(f"{OUT}/selected_mask.npy", mask)
shutil.make_archive("results_lab07", "zip", OUT)
try:
    from google.colab import files
    files.download("results_lab07.zip")
except ImportError:
    print("Colab ke bahar hai, zip folder mein save ho gayi.")